# DATT — repeatable Colab startup
Connect a T4 runtime, deploy the current source to `/content/DATT`, then run Cells 1–13 in order.
The notebook never pulls, resets, patches business code, prints credentials, or runs acceptance fixtures.
Existing source, compatible dependencies, model caches and a healthy CLI backend are reused.
After a failed cell, resolve the printed blocker and rerun from Cell 1. A missing deployment or incompatible
installed dependency stops startup. Email configuration is optional to load, but a running notification
worker is required for full SYSTEM_READY. No real camera is activated by this notebook.


In [ ]:
# CELL 1 — Runtime Preflight (no installation)
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path('/content/DATT')
startup = None
print('python=' + sys.version.split()[0])
print('cwd=' + str(Path.cwd()))
probe = subprocess.run([sys.executable, '-c', "import torch,json; print(json.dumps(dict(gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none', cuda=torch.version.cuda, available=torch.cuda.is_available())))"], capture_output=True, text=True, timeout=60)
if probe.returncode:
    raise RuntimeError('PyTorch/CUDA preflight failed. Select a T4 runtime; diagnostic output withheld.')
runtime = json.loads(probe.stdout)
print('gpu=' + runtime['gpu'])
print('cuda=' + str(runtime['cuda']))
if not runtime['available'] or not str(runtime['cuda']).startswith('12.'):
    raise RuntimeError('DATT requires a CUDA 12 GPU runtime. Select T4 before continuing.')
runtime = dict(runtime='colab', gpu=runtime['gpu'], cuda=runtime['cuda'], runtime_pid=os.getpid())


In [ ]:
# CELL 2 — Source Setup (reuse only; no git pull/reset)
if 'runtime' not in globals() or not isinstance(runtime, dict) or runtime.get('runtime') != 'colab':
    raise RuntimeError('Run Cell 1 successfully first.')
for name in ('scripts/datt.py', 'scripts/colab_startup.py', 'scripts/colab_install.py', 'src/main.py', 'src/db/alembic.ini'):
    if not (ROOT / name).is_file():
        raise RuntimeError('Deploy the current DATT source to /content/DATT; required file missing: ' + name)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.colab_startup import Startup
startup = Startup(runtime)
print('source=reused')
print('commit=' + startup.report['commit'])


In [ ]:
# CELL 3 — Dependencies
startup.run(3)


In [ ]:
# CELL 4 — Environment Configuration
startup.run(4)


In [ ]:
# CELL 5 — Persistence Preflight
startup.run(5)


In [ ]:
# CELL 6 — Models / GPU Runtime
startup.run(6)


In [ ]:
# CELL 7 — Database Migration
startup.run(7)


In [ ]:
# CELL 8 — Stop Stale DATT Processes
startup.run(8)


In [ ]:
# CELL 9 — Start DATT
startup.run(9)


In [ ]:
# CELL 10 — Health Check
startup.run(10)


In [ ]:
# CELL 11 — Application URLs
startup.run(11)


In [ ]:
# CELL 12 — Functional Smoke Test
startup.run(12)


In [ ]:
# CELL 13 — Ready State
startup.run(13)
